In [1]:
from shapely.geometry.polygon import LinearRing, Polygon, LineString, Point
import geopy
from geopy import *
from geopy.distance import great_circle,GreatCircleDistance
import geopandas
import pyproj
import json
import folium

import logging
#from userinputs import *
import pandas as pd
import os
import glob
import geopy
from geopy import distance
from tqdm.notebook import tqdm
import numpy as np
from sklearn.neighbors import BallTree, KDTree
import warnings
warnings.filterwarnings('ignore')

In [2]:
SITE_LOCATION_LONGITUDE = 'Longitude'
SITE_LOCATION_LATITUDE = 'Latitude'
FREQLAYER_COLUMN = 'DL EARFCN'
CELL_AZIMUTH = 'Azimuth'
CELL_NAME = 'Cell Name'
SITE_NAME = 'Site Name'

In [3]:
path_csv = "C:/IMAGE/Foliage/image_api_docs/Sitedata_1.csv"
path_out = "C:/IMAGE/Foliage/image_api_docs/shapefiles/"

In [4]:
def getSectorPolyList(lat, lon, azimuth, hpbw, dist):
    cell_poly=np.nan
    try:
        origin = geopy.Point(lat, lon)
        destination1 = GreatCircleDistance(
            meters=dist).destination(
            origin, azimuth - hpbw)
        lat1, lon1 = destination1.latitude, destination1.longitude

        destination1_1 = GreatCircleDistance(meters=dist).destination(
            origin, azimuth - hpbw + int(hpbw / 2))
        lat1_1, lon1_1 = destination1_1.latitude, destination1_1.longitude
        destination3 = GreatCircleDistance(
            meters=dist).destination(
            origin, azimuth)
        lat3, lon3 = destination3.latitude, destination3.longitude
        destination2_1 = GreatCircleDistance(meters=dist).destination(
            origin, azimuth + hpbw - int(hpbw / 2))
        lat2_1, lon2_1 = destination2_1.latitude, destination2_1.longitude

        destination2 = GreatCircleDistance(
            meters=dist).destination(
            origin, azimuth + hpbw)
        lat2, lon2 = destination2.latitude, destination2.longitude
        #lst=[[lon, lat],[lon1, lat1],[lon1_1, lat1_1],[lon3, lat3],[lon2_1, lat2_1],[lon2, lat2]]

        lst = [[lon, lat], [lon1, lat1], [lon1_1, lat1_1], [
            lon3, lat3], [lon2_1, lat2_1], [lon2, lat2]]
        #lst=[[Lon2X_(lon), Lat2Y_(lat)],[Lon2X_(lon1), Lat2Y_(lat1)],[Lon2X_(lon2), Lat2Y_(lat2)]]
        # return lst
        cell_poly = Polygon((lst))

    except Exception as e:
        cell_poly=np.nan
    return cell_poly

In [5]:
df = pd.read_csv(path_csv,sep=',')
df.head(2)

,Site Name,Cell Name,Latitude,Longitude,dlChannelBandwidth,DL EARFCN,PCI,Tx Power,SS EPRE,Electrical Tilt,...,Coverage Objective RSRP Threshold,Coverage Objective Percentile Threshold,Coverage Swap Percentile Threshold,Prediction Area,Prediction Category,Spectrum Band,Carrier No,OnAir,RTT Target(ms),Antenna_Tx
0,I-MH-ABAD-ENB-6000,I-MH-ABAD-ENB-6000_1,19.271819,72.88849,100,634080,6,55.05,19.9,4,...,-115,0.95,0.5,5G_3500_MHz_Maharashtra,TW,N3500,1,On,30,32
1,I-MH-ABAD-ENB-6000,I-MH-ABAD-ENB-6000_2,19.271819,72.88849,100,634080,7,55.05,19.9,4,...,-115,0.95,0.5,5G_3500_MHz_Maharashtra,TW,N3500,1,On,30,32


In [6]:
channel_list = sorted(list(set(df[FREQLAYER_COLUMN].tolist())))
band_len={}
bandcolor={}
point_radius={}
l=1000
r= 1
scale=1
for idx,f in enumerate(channel_list):
    band_len[f]=l
    # bandcolor[f] = colors[np.random.random_integers(low=0,high=len(colors)-1)]
    point_radius[f]=r/(idx+1)
    l=l+15

In [7]:
geometry_sitedata=[getSectorPolyList(lat, lon, azimuth, 30, band_len.get(band)*scale)
            for lat,lon,azimuth,band in zip(df[SITE_LOCATION_LATITUDE].tolist(),
                                            df[SITE_LOCATION_LONGITUDE].tolist(),
                                            df[CELL_AZIMUTH].tolist(),
                                            df[FREQLAYER_COLUMN].tolist())]

In [8]:
gdf_sitedata = geopandas.GeoDataFrame(df, geometry=geometry_sitedata)
gdf_sitedata = gdf_sitedata.set_crs(pyproj.CRS.from_epsg(4326), inplace=True)

In [9]:
sites_unique = gdf_sitedata[SITE_NAME].unique()
print(sites_unique)

['I-MH-ABAD-ENB-6000' 'I-MH-ABAD-ENB-6001' 'I-MH-ABAD-ENB-6002'
 'I-MH-ABAD-ENB-6004' 'I-MH-ABAD-ENB-6005']


In [10]:
for site in sites_unique:
    gdf_site = gdf_sitedata[gdf_sitedata[SITE_NAME]==site]
    gdf_site.to_file(filename=path_out+site+'.shp', driver='ESRI Shapefile')

In [11]:
#gdf_sitedata.to_file(filename=path_out+'dPolygons.shp', driver='ESRI Shapefile')